# 📥 CMF PDF Downloader — SITEX

Automatically discovers and downloads **'Etats financiers au 31/12'** annual reports for **Sté. INDUSTRIELLE DES TEXTILES - SITEX** from the CMF website.

- Scrapes the official CMF consultation page for SITEX
- Detects PDF links via `<img class="file-icon" src=".../application-pdf.png">` markers
- Covers years **2015 → 2026**
- Skips already-downloaded files

---

## 1. Install Dependencies

In [1]:
!pip install requests beautifulsoup4 -q
print("✅ Dependencies ready.")

✅ Dependencies ready.


## 2. Imports

In [2]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import display
import pandas as pd

print("✅ Imports done.")

✅ Imports done.


## 3. Configuration

> **Edit this cell to change the output folder or year range.**

In [3]:
# ─── USER CONFIGURATION ───────────────────────────────────────────────────────

OUTPUT_DIR  = "CMF_SITEX_PDFs"   # Folder where PDFs will be saved
BASE_DOMAIN = "https://www.cmf.tn"
YEAR_START  = 2015
YEAR_END    = 2026

# CMF consultation page for SITEX
CMF_PAGE_URL = (
    "https://www.cmf.tn/?q=consultation-des-tats-financier-des-soci-t-s-faisant-ape"
    "&field_societesape_value=St%C3%A9.+INDUSTRIELLE+DES+TEXTILES+-+SITEX+-"
)

# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output folder : {os.path.abspath(OUTPUT_DIR)}")
print(f"Year range    : {YEAR_START} → {YEAR_END}")
print(f"Source URL    : {CMF_PAGE_URL}")

Output folder : C:\Users\Lenovo\CMF_SITEX_PDFs
Year range    : 2015 → 2026
Source URL    : https://www.cmf.tn/?q=consultation-des-tats-financier-des-soci-t-s-faisant-ape&field_societesape_value=St%C3%A9.+INDUSTRIELLE+DES+TEXTILES+-+SITEX+-


## 4. Filters & Headers

Only rows whose label matches **'Etats financiers au 31/12'** and whose year falls within the configured range are kept.

In [4]:
# Label filter — same logic as the original notebook
LABEL_PATTERN = re.compile(
    r"etats?\s+financiers?\s+(?:consolid[ée]s?\s+)?au\s+31/12",
    re.IGNORECASE,
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    )
}

print("✅ Filters & headers configured.")

✅ Filters & headers configured.


## 5. Discovery — Scrape the SITEX Page for PDF Links

Fetches the CMF page and finds every `<img class="file-icon" src=".../application-pdf.png">` element — the same HTML marker used on CMF pages to signal a downloadable PDF. Extracts the parent `<a>` href and applies year + label filters.

In [5]:
def extract_year_from_text(text: str) -> int | None:
    """Pull a 4-digit year (2000-2099) out of a string, or None."""
    m = re.search(r"\b(20\d{2})\b", text)
    return int(m.group(1)) if m else None


def scrape_pdf_links(page_url: str) -> list[dict]:
    """
    Fetch *page_url* and return a list of dicts:
        {"url": str, "label": str, "year": int | None, "filename": str}
    for every <img class="file-icon" src=".../application-pdf.png"> found.
    """
    resp = requests.get(page_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    results = []
    seen_urls = set()

    # Find every file-icon image that points to the PDF icon
    for img in soup.find_all("img", class_="file-icon"):
        src = img.get("src", "")
        if "application-pdf" not in src:
            continue  # skip non-PDF icons

        # The PDF link is the closest ancestor <a> tag
        anchor = img.find_parent("a")
        if anchor is None:
            continue

        href = anchor.get("href", "")
        if not href:
            continue

        full_url = urljoin(BASE_DOMAIN, href)
        if full_url in seen_urls:
            continue
        seen_urls.add(full_url)

        # Walk up the DOM to find the nearest block with descriptive text
        label = ""
        for parent in anchor.parents:
            text = parent.get_text(" ", strip=True)
            if len(text) > 5 and text != label:
                label = text
                break

        filename = href.rstrip("/").split("/")[-1]
        year = extract_year_from_text(label) or extract_year_from_text(filename)

        results.append({
            "url": full_url,
            "label": label,
            "year": year,
            "filename": filename,
        })

    return results


print("🔍 Fetching CMF SITEX page and discovering PDF links...\n")

all_links = scrape_pdf_links(CMF_PAGE_URL)
print(f"   Total PDF links found on page : {len(all_links)}")

# ── Apply filters ──────────────────────────────────────────────────────────────
records = []  # (year, doc_type, url, filename)

for link in all_links:
    year  = link["year"]
    label = link["label"]

    # Year range filter
    if year is None or not (YEAR_START <= year <= YEAR_END):
        continue

    # Label filter — keep only "Etats financiers au 31/12" entries
    if not LABEL_PATTERN.search(label):
        continue

    doc_type = (
        "Etats financiers consolidés au 31/12"
        if re.search(r"consolid", label, re.IGNORECASE)
        else "Etats financiers au 31/12"
    )

    records.append((year, doc_type, link["url"], link["filename"]))
    print(f"  [{year}] ✓  {link['filename']}")

# Deduplicate by URL
records = list({r[2]: r for r in records}.values())

print(f"\n✅ Discovery complete — {len(records)} document(s) found.")

🔍 Fetching CMF SITEX page and discovering PDF links...

   Total PDF links found on page : 20
  [2025] ✓  sitex_efd311224.pdf
  [2024] ✓  sitex_efd311223.pdf
  [2023] ✓  sitex_efd311222.pdf
  [2022] ✓  sitex_efd311221.pdf
  [2021] ✓  sitex-efd-3112-2020.pdf
  [2020] ✓  sitex_efd311219.pdf
  [2019] ✓  sitex_efd311218.pdf
  [2018] ✓  sitex_efd311217.pdf
  [2017] ✓  sitex_efd311216.pdf
  [2016] ✓  sitex_efd311215.pdf

✅ Discovery complete — 10 document(s) found.


## 6. Preview Discovered Documents

In [6]:
if records:
    preview_df = pd.DataFrame(
        [(y, dt, fn) for y, dt, url, fn in sorted(records, reverse=True)],
        columns=["Year", "Document Type", "Filename"],
    )
    display(preview_df)
else:
    print("⚠️  No documents found.")
    print("    CMF may publish financial statements inside Bulletin Officiel PDFs")
    print("    instead of separate files. Check the website manually.")

,Year,Document Type,Filename
0,2025,Etats financiers au 31/12,sitex_efd311224.pdf
1,2024,Etats financiers au 31/12,sitex_efd311223.pdf
2,2023,Etats financiers au 31/12,sitex_efd311222.pdf
3,2022,Etats financiers au 31/12,sitex_efd311221.pdf
4,2021,Etats financiers au 31/12,sitex-efd-3112-2020.pdf
5,2020,Etats financiers au 31/12,sitex_efd311219.pdf
6,2019,Etats financiers au 31/12,sitex_efd311218.pdf
7,2018,Etats financiers au 31/12,sitex_efd311217.pdf
8,2017,Etats financiers au 31/12,sitex_efd311216.pdf
9,2016,Etats financiers au 31/12,sitex_efd311215.pdf


## 7. Download PDFs

Files already present in the output folder are skipped automatically.

In [7]:
if not records:
    print("Nothing to download — run the discovery cell first.")
else:
    downloaded = []
    failed     = []

    for year, doc_type, url, filename in sorted(records, key=lambda x: x[0], reverse=True):
        save_path = os.path.join(OUTPUT_DIR, filename)

        # ── Already on disk ────────────────────────────────────────────────
        if os.path.exists(save_path):
            size_kb = os.path.getsize(save_path) // 1024
            print(f"[{year}] ⏭  Already exists: {filename} ({size_kb} KB)")
            downloaded.append((year, filename, size_kb, "skipped"))
            continue

        # ── Download ───────────────────────────────────────────────────────
        print(f"[{year}] ⬇  Downloading: {filename} ...", end=" ", flush=True)
        try:
            r = requests.get(url, headers=HEADERS, timeout=60, stream=True)
            r.raise_for_status()

            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

            size_kb = os.path.getsize(save_path) // 1024
            print(f"✓ saved ({size_kb} KB)")
            downloaded.append((year, filename, size_kb, "downloaded"))

        except Exception as e:
            print(f"✗ FAILED")
            failed.append((year, filename, str(e)))

    print(f"\n{'='*70}")
    print(f"✅ Done: {len(downloaded)} file(s) processed, {len(failed)} failure(s).")

[2025] ⬇  Downloading: sitex_efd311224.pdf ... ✓ saved (10799 KB)
[2024] ⬇  Downloading: sitex_efd311223.pdf ... ✓ saved (1977 KB)
[2023] ⬇  Downloading: sitex_efd311222.pdf ... ✓ saved (1272 KB)
[2022] ⬇  Downloading: sitex_efd311221.pdf ... ✓ saved (215 KB)
[2021] ⬇  Downloading: sitex-efd-3112-2020.pdf ... ✓ saved (4429 KB)
[2020] ⬇  Downloading: sitex_efd311219.pdf ... ✓ saved (1290 KB)
[2019] ⬇  Downloading: sitex_efd311218.pdf ... ✓ saved (141 KB)
[2018] ⬇  Downloading: sitex_efd311217.pdf ... ✓ saved (1154 KB)
[2017] ⬇  Downloading: sitex_efd311216.pdf ... ✓ saved (606 KB)
[2016] ⬇  Downloading: sitex_efd311215.pdf ... ✓ saved (184 KB)

✅ Done: 10 file(s) processed, 0 failure(s).
